In [1]:
from pathlib import Path
import sys

_here = Path.cwd().resolve()
_candidates = (_here, *_here.parents)
REPO_ROOT = next((candidate for candidate in _candidates if (candidate / "m33_pipeline").is_dir()), None)
if REPO_ROOT is None:
    raise RuntimeError(f"Could not locate repo root from {Path.cwd()}")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from m33_pipeline.notebook_setup import prepare_notebook

REPO_ROOT = prepare_notebook(REPO_ROOT)
print(f"Notebook working directory set to: {REPO_ROOT}")

Notebook working directory set to: /Users/emmajarvis/Documents/SIGNALS/M33/PAPER1


In [10]:
try:
    flux_method
except NameError:
    flux_method = 'summed_map'
flux_method = 'summed_map'
try:
    dig_mode
except NameError:
    dig_mode = 'dig_subtracted'
dig_mode = 'dig_subtracted'
print(f"Using flux_method={flux_method}, dig_mode={dig_mode}")


from m33_pipeline.config import get_derived_config
from m33_pipeline.derived import (
    add_clustering_metrics,
    add_electron_density,
    add_logU_KK04,
    add_metallicity_columns,
    add_metallicity_error_columns,
    add_boundary_source_flags,
    add_primary_overlap_flags,
    add_peak_region_properties,
    add_symmetry_class,
    add_thermal_pressure,
    merge_field_flux_catalogs,
    write_clustering_outputs,
    write_combined_catalog,
    write_derived_stage_catalog,
    write_total_flux_catalog,
)
from m33_pipeline.validate import validate_total_catalog


Using flux_method=summed_map, dig_mode=dig_subtracted


# Merge per-field flux catalogs


In [11]:
derived_config = get_derived_config()
all_catalog = merge_field_flux_catalogs(method=flux_method, dig_mode=dig_mode)
all_catalog = add_primary_overlap_flags(
    all_catalog,
    match_radius_px=5.0,
    boundary_overlap=True,
    boundary_min_overlap_pixels=1,
    boundary_valid_bounds=(50, 2000, 50, 2000),
)
all_catalog = add_boundary_source_flags(all_catalog, max_zoi_pc=100)
n_overlap_groups = int(all_catalog['is_duplicate_overlap'].fillna(False).groupby(all_catalog['duplicate_group_id']).any().sum()) if 'duplicate_group_id' in all_catalog.columns else 0
n_duplicate_rows = int(all_catalog['is_duplicate_overlap'].fillna(False).sum()) if 'is_duplicate_overlap' in all_catalog.columns else 0
n_non_primary = int((~all_catalog['primary'].fillna(True)).sum()) if 'primary' in all_catalog.columns else 0
n_wr_flagged = int(all_catalog['has_wr_in_boundary'].fillna(False).sum()) if 'has_wr_in_boundary' in all_catalog.columns else 0
n_snr_flagged = int(all_catalog['has_snr_in_boundary'].fillna(False).sum()) if 'has_snr_in_boundary' in all_catalog.columns else 0
print(f"Overlap duplicate groups: {n_overlap_groups}")
print(f"Rows involved in overlap duplicates: {n_duplicate_rows}")
print(f"Rows flagged as non-primary: {n_non_primary}")
print(f"Regions containing WR stars: {n_wr_flagged}")
print(f"Regions containing SNRs: {n_snr_flagged}")
if n_overlap_groups > 0:
    dup_preview_cols = [c for c in ['field', 'region_id', 'duplicate_group_id', 'primary', 'primary_rank', 'primary_region_id', 'duplicate_score_sum_snr'] if c in all_catalog.columns]
    print(all_catalog.loc[all_catalog['is_duplicate_overlap'], dup_preview_cols].sort_values(['duplicate_group_id', 'primary_rank']).head(20).to_string(index=False))
output_path = write_total_flux_catalog(all_catalog, method=flux_method, dig_mode=dig_mode)
print("Flux method:", flux_method)
print("DIG mode:", dig_mode)
print("Combined catalog shape:", all_catalog.shape)
print("Saved combined catalog to:", output_path)
# validate_total_catalog(all_catalog)


Overlap duplicate groups: 141
Rows involved in overlap duplicates: 661
Rows flagged as non-primary: 520
Regions containing WR stars: 23
Regions containing SNRs: 52
field  region_id duplicate_group_id  primary  primary_rank  primary_region_id  duplicate_score_sum_snr
   NE        709           dup_0001     True             1                709               511.281994
   F5          2           dup_0001    False             2                709               230.767276
   NE        711           dup_0002     True             1                711              1226.511180
   F5          3           dup_0002    False             2                711              1038.820828
   NE        706           dup_0002    False             3                711               672.006127
   NE        720           dup_0002    False             4                711               584.039061
   NE        722           dup_0002    False             5                711               357.523017
   NE       

# Add ionization parameter


In [12]:
cat = add_logU_KK04(all_catalog.copy(), n_mc=derived_config.logu_n_mc, seed=123)
derived_output_path = write_derived_stage_catalog(cat, "ionization_parameter", method=flux_method, dig_mode=dig_mode)
print("Number of columns:", len(cat.columns))
print("Saved combined catalog with ionization parameter:", derived_output_path)


Number of columns: 264
Saved combined catalog with ionization parameter: /Users/emmajarvis/Documents/SIGNALS/M33/PAPER1/CATALOGS/flux_catalogs/summed_map/dig_subtracted/derived/total_flux_catalog_ionization_parameter.csv


# Add electron density


In [13]:
df = add_electron_density(cat.copy(), n_mc=derived_config.density_n_mc)
df = add_thermal_pressure(
    df,
    T_e=derived_config.electron_temperature_K,
    particle_factor=derived_config.ionized_gas_particle_factor,
)
df = add_peak_region_properties(
    df,
    distance_mpc=derived_config.m33_distance_mpc,
    te_default=derived_config.electron_temperature_K,
    n_mc=derived_config.density_n_mc,
)
derived_output_path = write_derived_stage_catalog(df, "density", method=flux_method, dig_mode=dig_mode)
print("Number of columns:", len(df.columns))
print("Saved combined catalog with electron densities:", derived_output_path)


Number of columns: 320
Saved combined catalog with electron densities: /Users/emmajarvis/Documents/SIGNALS/M33/PAPER1/CATALOGS/flux_catalogs/summed_map/dig_subtracted/derived/total_flux_catalog_density.csv


# Add symmetry classification


In [14]:
df = add_symmetry_class(df.copy())
derived_output_path = write_derived_stage_catalog(df, "other_derived", method=flux_method, dig_mode=dig_mode)
print("Symmetry classification counts:")
print(df["symmetry_class"].value_counts())
print("Number of columns:", len(df.columns))
print("Saved combined catalog with other derived properties:", derived_output_path)


Symmetry classification counts:
symmetry_class
asymmetric    5247
symmetric     1176
Name: count, dtype: int64
Number of columns: 321
Saved combined catalog with other derived properties: /Users/emmajarvis/Documents/SIGNALS/M33/PAPER1/CATALOGS/flux_catalogs/summed_map/dig_subtracted/derived/total_flux_catalog_other_derived.csv


# Add metallicity calibrations


In [15]:
df = add_metallicity_columns(df.copy())
df = add_metallicity_error_columns(df.copy(), n_mc=derived_config.metallicity_n_mc, seed=123)
derived_output_path = write_derived_stage_catalog(df, "metallicity", method=flux_method, dig_mode=dig_mode)
combined_output_path = write_combined_catalog(df, method=flux_method, dig_mode=dig_mode)
print("Number of columns:", len(df.columns))
print("Saved combined catalog with metallicities:", derived_output_path)
print("Saved final combined catalog:", combined_output_path)


Number of columns: 373
Saved combined catalog with metallicities: /Users/emmajarvis/Documents/SIGNALS/M33/PAPER1/CATALOGS/flux_catalogs/summed_map/dig_subtracted/derived/total_flux_catalog_metallicity.csv
Saved final combined catalog: /Users/emmajarvis/Documents/SIGNALS/M33/PAPER1/CATALOGS/flux_catalogs/summed_map/dig_subtracted/total_flux_catalog_combined.csv


# Add deprojected clustering metrics


In [16]:
clustered_df, global_stats, ripley_df, pcf_df = add_clustering_metrics(df.copy())
outputs = write_clustering_outputs(clustered_df, global_stats, ripley_df, pcf_df, method=flux_method, dig_mode=dig_mode)
print("Saved catalog:", outputs["catalog"])
print("Saved global stats:", outputs["global"])
print("Saved Ripley profile:", outputs["ripley"])
print("Saved pair-correlation profile:", outputs["pcf"])


Saved catalog: /Users/emmajarvis/Documents/SIGNALS/M33/PAPER1/CATALOGS/flux_catalogs/summed_map/dig_subtracted/derived/total_flux_catalog_clustering.csv
Saved global stats: /Users/emmajarvis/Documents/SIGNALS/M33/PAPER1/CATALOGS/flux_catalogs/summed_map/dig_subtracted/derived/clustering_global_statistics.csv
Saved Ripley profile: /Users/emmajarvis/Documents/SIGNALS/M33/PAPER1/CATALOGS/flux_catalogs/summed_map/dig_subtracted/derived/clustering_ripley_profile.csv
Saved pair-correlation profile: /Users/emmajarvis/Documents/SIGNALS/M33/PAPER1/CATALOGS/flux_catalogs/summed_map/dig_subtracted/derived/clustering_pair_correlation_profile.csv
